In [16]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import sys
import re
import os

root_dir = "/work/yufeng/2022/enzyme_specificity"

sys.path.append(f"{root_dir}/src")

# Download enzyme active info from brenda
See Dataset.utils (download_uniprot_file)

# Create the reaction features
See Dataset.utils (get_reaction_feature)

In [14]:
reaction_df = pd.read_csv(f"{root_dir}/data/full_brenda/reaction copy.csv", sep=',')

drop_idx = []
for index, (reaction, substrate) in enumerate(zip(reaction_df['reactions'].values.tolist(), reaction_df['substrates'].values.tolist())):
    if len(substrate) > 275:
        drop_idx.append(index)

reaction_df = reaction_df.drop(drop_idx).reset_index(drop=True)
reaction_df.to_csv(f"{root_dir}/data/full_brenda/reaction.csv", sep=',', index=False)

# Create the enzyme features
See Dataset.utils (get_enzyme_feature)

# Create positive samples dataset

In [17]:
df = pd.read_csv(f"{root_dir}/data/brenda/data.csv", sep=',').dropna(subset=['left', 'right', 'uniprot', 'ecnumber'])
enzyme_df = pd.read_csv(f"{root_dir}/data/full_brenda/enzymes.csv", sep=',')
reaction_df = pd.read_csv(f"{root_dir}/data/full_brenda/reaction.csv", sep=',')

uniprot_dict = {uniprot: index for index, uniprot in enumerate(enzyme_df['uniprots'].values.tolist())}
reaction_dict = {}

for index, (reaction, substrate) in enumerate(zip(reaction_df['reactions'].values.tolist(), reaction_df['substrates'].values.tolist())):
    reaction_dict[reaction] = index

data = {
    'reaction': [],
    'enzyme': [],
    'ecnumber': []
}

for left, right, uniprot, ecnumber in zip(df['left'], df['right'], df['uniprot'], df['ecnumber']):
    if uniprot in uniprot_dict and left + '>>' + right in reaction_dict:
        data['reaction'].append(reaction_dict[left + '>>' + right])
        data['enzyme'].append(uniprot_dict[uniprot])
        data['ecnumber'].append(ecnumber)

data = pd.DataFrame(data).sample(frac=1).reset_index()
data.to_csv(f"{root_dir}/data/full_brenda/positive_data.csv", index=False)


# Calculate max_length of reaction

In [18]:
import lmdb
import pickle

reaction_save_lmdb_path = f"{root_dir}/data/full_brenda/reaction_features.lmdb"
db = lmdb.open(
    reaction_save_lmdb_path,
    map_size=10*(1024*1024*1024),   # 10GB
    create=False,
    subdir=False,
    readonly=True,
    lock=False,
    readahead=False,
    meminit=False,
)
with db.begin() as txn:
    keys = list(txn.cursor().iternext(values=False))

max_n_atoms = 0

for key in keys:
     with db.begin(write=False, buffers=True) as txn:
        # key = str(key).encode()
        value = txn.get(key)
        if value is None:
            raise KeyError
        data = pickle.loads(value)
        max_n_atoms = max(max_n_atoms, data['element'].shape[0])
        # break
print(max_n_atoms)

135


# Create grover embedding

In [19]:
# 1. Tou Tou is going to create smile only csv
import pandas as pd
import os

df = pd.read_csv(f"{root_dir}/data/full_brenda/reaction.csv", sep=',')
results = [smile for smile in df['substrates']]

for smile in df['substrates']:
    if len(smile) < 275:
        results.append(smile)
    
data = {
    "substrates": results
}
data = pd.DataFrame(data)
data.to_csv(f"{root_dir}/data/full_brenda/reaction_smiles.csv", index=False)

In [20]:
# 2. Get npz feature
os.system(f"python {root_dir}/src/other_softwares/grover_software/scripts/save_features.py --data_path {root_dir}/data/full_brenda/reaction_smiles.csv  \
                                --save_path {root_dir}/data/full_brenda/reaction.npz   \
                                --features_generator fgtasklabel \
                                --restart")

[00:03:55] WARNING: not removing hydrogen atom without neighbors
[00:03:55] WARNING: not removing hydrogen atom without neighbors
[00:03:55] WARNING: not removing hydrogen atom without neighbors
[00:03:55] WARNING: not removing hydrogen atom without neighbors
[00:03:55] WARNING: not removing hydrogen atom without neighbors
[00:03:55] WARNING: not removing hydrogen atom without neighbors
[00:03:55] WARNING: not removing hydrogen atom without neighbors
[00:03:55] WARNING: not removing hydrogen atom without neighbors
[00:03:55] WARNING: not removing hydrogen atom without neighbors
[00:03:56] WARNING: not removing hydrogen atom without neighbors
[00:03:56] WARNING: not removing hydrogen atom without neighbors
[00:03:56] WARNING: not removing hydrogen atom without neighbors
[00:03:56] WARNING: not removing hydrogen atom without neighbors
[00:03:56] WARNING: not removing hydrogen atom without neighbors
[00:03:56] WARNING: not removing hydrogen atom without neighbors
[00:03:58] WARNING: not r

0

In [21]:
# 3. Get build vocab
os.system(f"python {root_dir}/src/other_softwares/grover_software/scripts/build_vocab.py --data_path {root_dir}/data/full_brenda/reaction_smiles.csv \
                             --vocab_save_folder {root_dir}/data/full_brenda/grover_vocab  \
                             --dataset_name brenda")

                            

Building atom vocab from file: /work/yufeng/2022/enzyme_specificity/data/full_brenda/reaction_smiles.csv


100000it [00:19, 5156.83it/s]                          
  0%|          | 0/80143 [00:00<?, ?it/s]

atom vocab size 462
Building bond vocab from file: /work/yufeng/2022/enzyme_specificity/data/full_brenda/reaction_smiles.csv


100000it [01:26, 1151.32it/s]                         


bond vocab size 573


0

In [22]:
# 4. Get fingerprint
print(f"CUDA_VISIBLE_DEVICES=1 python main.py fingerprint --data_path {root_dir}/data/full_brenda/reaction_smiles.csv --features_path {root_dir}/data/full_brenda/reaction.npz --checkpoint_path {root_dir}/data/pretrain_model/grover_large.pt --fingerprint_source both --output {root_dir}/data/full_brenda/fingerprint.npz --save_lmdb_path {root_dir}/data/full_brenda/grover_fingerprint.lmdb --fingerprint_source both")

CUDA_VISIBLE_DEVICES=1 python main.py fingerprint --data_path /work/yufeng/2022/enzyme_specificity/data/full_brenda/reaction_smiles.csv --features_path /work/yufeng/2022/enzyme_specificity/data/full_brenda/reaction.npz --checkpoint_path /work/yufeng/2022/enzyme_specificity/data/pretrain_model/grover_large.pt --fingerprint_source both --output /work/yufeng/2022/enzyme_specificity/data/full_brenda/fingerprint.npz --save_lmdb_path /work/yufeng/2022/enzyme_specificity/data/full_brenda/grover_fingerprint.lmdb --fingerprint_source both


# Create morgan embedding

In [24]:
from rdkit.Chem import AllChem
from rdkit import Chem

path = f"{root_dir}/data/full_brenda/reaction_smiles.csv"
df = pd.read_csv(path, sep=',')
results = []
for smile in df['substrates']:
    m1 = Chem.MolFromSmiles(smile)
    result = np.array(AllChem.GetMorganFingerprintAsBitVect(m1,2,nBits=1024))
    results.append(result)
np.save(f"{root_dir}/data/full_brenda/morgan_fingerprint.npy", np.array(results))

[00:30:01] WARNING: not removing hydrogen atom without neighbors
[00:30:01] WARNING: not removing hydrogen atom without neighbors
[00:30:01] WARNING: not removing hydrogen atom without neighbors
[00:30:01] WARNING: not removing hydrogen atom without neighbors
[00:30:02] WARNING: not removing hydrogen atom without neighbors
[00:30:02] WARNING: not removing hydrogen atom without neighbors
[00:30:02] WARNING: not removing hydrogen atom without neighbors
[00:30:02] WARNING: not removing hydrogen atom without neighbors
[00:30:02] WARNING: not removing hydrogen atom without neighbors
[00:30:11] WARNING: not removing hydrogen atom without neighbors
[00:30:11] WARNING: not removing hydrogen atom without neighbors
[00:30:11] WARNING: not removing hydrogen atom without neighbors
[00:30:11] WARNING: not removing hydrogen atom without neighbors
[00:30:11] WARNING: not removing hydrogen atom without neighbors
[00:30:11] WARNING: not removing hydrogen atom without neighbors
[00:30:24] WARNING: not r

# Select data points for halogenase

In [23]:
from Datasets.utils import generate_negative_sample
from easydict import EasyDict
ecnumbers = ["1.11.1.10", "1.11.1.-", "1.11.1.18", "1.14.19.9", "1.14.14.-","1.14.19.56", "1.14.19.-", "1.14.99.-", "1.14.19.49", "3.8.1.1", "1.14.20.-", "2.5.1.94", "2.2.1.6", "2.5.1.63", "3.13.1.8", "2.5.1.-"]
n_digit = 0

for n_digit in range(0, 5):
    config = {
        "data": {
            "sampling": {
                "same_digits": 0,
                "num_negative_enzyme": 2
            }
        }
    }
    config = EasyDict(config)

    def get_number_same_digits(ecnumber1, ecnumber2):
        ecnumber_digits1 = ecnumber1.split(".")
        ecnumber_digits2 = ecnumber2.split(".")
        for index, (digit1, digit2) in enumerate(zip(ecnumber_digits1, ecnumber_digits2)):
            if digit1 == '-' or digit2 == '-':
                continue
            if digit1 != digit2:
                return index
        return 4

    df = pd.read_csv(f"{root_dir}/data/full_brenda/positive_data.csv", sep=',')

    drop_indexs = []
    for index, ecnumber in enumerate(df['ecnumber']):
        flag = False
        for ecnumber in ecnumbers:
            if get_number_same_digits(ecnumber, df['ecnumber'][index]) >= n_digit:
                flag = True
                break
        if not flag or n_digit == 4:
            drop_indexs.append(index)
    df = df.drop(drop_indexs).reset_index(drop=True)
    df = generate_negative_sample(config, df)
    df.to_csv(f"{root_dir}/data/halogenase/full_brenda_data_{n_digit}.csv", sep=',', index=False)

/work/yufeng/miniconda/envs/revae/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 11796/11796 [00:00<00:00, 39421.54it/s]
0it [00:00, ?it/s]
